In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

In [2]:
# 데이터 불러오기
valid_vote_daily = pd.read_csv('valid_vote_daily.csv')
user_activation_retention = pd.read_csv('user_activation_retention.csv')
vote_event_analysis = pd.read_csv('vote_event_analysis.csv')

In [3]:
# =========================================================
# 초기 강도 / 초기 지속성 기준 선정
#
# 1) 1차 Activation 사용자 추출
# 2) 첫 Vote 후 24시간 Vote 횟수 계산
# 3) R1~R7 Vote 활동일 수 계산
# 4) 초기 강도 후보 기준 비교
# 5) 초기 지속성 후보 기준 비교
#
# 이 결과를 바탕으로 최종 기준:
# - 초기 강도 = 30회 이상
# - 초기 지속성 = 5일 이상

# ---------------------------------------------------------
# 1. 1차 Activation 사용자 추출
#    2023년 5월 가입 + 가입 후 48시간 내 Valid Vote 1회 이상
# ---------------------------------------------------------

may_activation_users = user_activation_retention[
    (user_activation_retention['variant'] == 'base') &
    (user_activation_retention['cohort_month'] == '2023-05') &
    (user_activation_retention['activation_48h_observed'] == True)
].copy()


# ---------------------------------------------------------
# 2. 초기 강도 계산을 위한
#    첫 Valid Vote 후 24시간 Vote 횟수 계산
# ---------------------------------------------------------

vote_event_analysis['vote_at_kst'] = pd.to_datetime(
    vote_event_analysis['vote_at_kst']
)

may_activation_users['first_vote_at'] = pd.to_datetime(
    may_activation_users['first_observed_valid_vote_at_kst']
)

vote_temp = (
    vote_event_analysis[
        vote_event_analysis['valid_base'] == True
    ]
    .merge(
        may_activation_users[['user_id', 'first_vote_at']],
        on='user_id',
        how='inner'
    )
)

vote_temp = vote_temp[
    (vote_temp['vote_at_kst'] >= vote_temp['first_vote_at']) &
    (
        vote_temp['vote_at_kst']
        < vote_temp['first_vote_at'] + pd.Timedelta(hours=24)
    )
]

first_vote_24h_count = (
    vote_temp
    .groupby('user_id')
    .size()
)

may_activation_users['first_vote_24h_votes'] = (
    may_activation_users['user_id']
    .map(first_vote_24h_count)
    .fillna(0)
    .astype(int)
)


# ---------------------------------------------------------
# 3. 초기 지속성 계산을 위한
#    R1~R7 Vote 활동일 수 계산
# ---------------------------------------------------------

r1_7_vote_days = (
    valid_vote_daily[
        (valid_vote_daily['variant'] == 'base') &
        (
            valid_vote_daily['user_id']
            .isin(may_activation_users['user_id'])
        ) &
        (valid_vote_daily['rday'].between(1, 7))
    ]
    .groupby('user_id')['rday']
    .nunique()
)

may_activation_users['vote_days_R1_7'] = (
    may_activation_users['user_id']
    .map(r1_7_vote_days)
    .fillna(0)
    .astype(int)
)


# ---------------------------------------------------------
# 4. Retention 비교용 사용자 플래그 생성
# ---------------------------------------------------------

def retained_users(start_day, end_day):
    return set(
        valid_vote_daily.loc[
            (valid_vote_daily['variant'] == 'base') &
            (
                valid_vote_daily['user_id']
                .isin(may_activation_users['user_id'])
            ) &
            (valid_vote_daily['rday'].between(start_day, end_day)),
            'user_id'
        ]
    )


# 초기 강도 기준 비교용
may_activation_users['R8_14_retained'] = (
    may_activation_users['user_id']
    .isin(retained_users(8, 14))
)

may_activation_users['R31_60_retained'] = (
    may_activation_users['user_id']
    .isin(retained_users(31, 60))
)

# 초기 지속성 기준 비교용
may_activation_users['W2_retained'] = (
    may_activation_users['user_id']
    .isin(retained_users(8, 14))
)

may_activation_users['W4_retained'] = (
    may_activation_users['user_id']
    .isin(retained_users(22, 28))
)

may_activation_users['W8_retained'] = (
    may_activation_users['user_id']
    .isin(retained_users(50, 56))
)


# =========================================================
# 5. 초기 강도 후보 기준 비교
# =========================================================

intensity_thresholds = [5, 10, 15, 20, 25, 30, 40, 50]

intensity_results = []

total_users = may_activation_users['user_id'].nunique()

for threshold in intensity_thresholds:

    high = (
        may_activation_users['first_vote_24h_votes']
        >= threshold
    )

    user_count = high.sum()
    user_rate = user_count / total_users * 100

    r8_14_gap = (
        may_activation_users.loc[
            high, 'R8_14_retained'
        ].mean()
        -
        may_activation_users.loc[
            ~high, 'R8_14_retained'
        ].mean()
    ) * 100

    r31_60_gap = (
        may_activation_users.loc[
            high, 'R31_60_retained'
        ].mean()
        -
        may_activation_users.loc[
            ~high, 'R31_60_retained'
        ].mean()
    ) * 100

    intensity_results.append({
        '기준': f'{threshold}회 이상',
        'Activation 사용자 수': user_count,
        'Activation 포함률': user_rate,
        'R8~R14 Retention 차이': r8_14_gap,
        'R31~R60 Retention 차이': r31_60_gap
    })


intensity_threshold_result = (
    pd.DataFrame(intensity_results)
    .round(2)
)


# =========================================================
# 6. 초기 지속성 후보 기준 비교
# =========================================================

persistence_thresholds = [3, 4, 5]

persistence_results = []

previous_count = None
previous_rate = None

for threshold in persistence_thresholds:

    persistent = (
        may_activation_users['vote_days_R1_7']
        >= threshold
    )

    user_count = persistent.sum()
    user_rate = user_count / total_users * 100

    # 직전 기준 대비 감소
    if previous_count is None:
        count_drop = np.nan
        rate_drop = np.nan
        relative_drop = np.nan

    else:
        count_drop = user_count - previous_count
        rate_drop = user_rate - previous_rate
        relative_drop = (
            (user_count - previous_count)
            / previous_count * 100
        )

    # W2 / W4 / W8 Retention 격차
    w2_gap = (
        may_activation_users.loc[
            persistent, 'W2_retained'
        ].mean()
        -
        may_activation_users.loc[
            ~persistent, 'W2_retained'
        ].mean()
    ) * 100

    w4_gap = (
        may_activation_users.loc[
            persistent, 'W4_retained'
        ].mean()
        -
        may_activation_users.loc[
            ~persistent, 'W4_retained'
        ].mean()
    ) * 100

    w8_gap = (
        may_activation_users.loc[
            persistent, 'W8_retained'
        ].mean()
        -
        may_activation_users.loc[
            ~persistent, 'W8_retained'
        ].mean()
    ) * 100

    persistence_results.append({
        '활동일 기준': f'{threshold}일+',
        '사용자': user_count,
        'Activated 비중': user_rate,
        '직전 대비 감소': count_drop,
        '비중 감소': rate_drop,
        '상대 감소율': relative_drop,
        'W2 격차': w2_gap,
        'W4 격차': w4_gap,
        'W8 격차': w8_gap
    })

    previous_count = user_count
    previous_rate = user_rate


persistence_threshold_result = (
    pd.DataFrame(persistence_results)
    .round(2)
)


# ---------------------------------------------------------
# 7. 기준 비교 결과 확인
# ---------------------------------------------------------

print('===== 초기 강도 기준 비교 =====')
display(intensity_threshold_result)

print('\n===== 초기 지속성 기준 비교 =====')
display(persistence_threshold_result)

===== 초기 강도 기준 비교 =====


,기준,Activation 사용자 수,Activation 포함률,R8~R14 Retention 차이,R31~R60 Retention 차이
0,5회 이상,3964,95.80,23.59,9.15
1,10회 이상,3704,89.51,19.08,6.68
2,15회 이상,3481,84.12,15.88,4.68
3,20회 이상,3237,78.23,12.45,4.84
4,25회 이상,3013,72.81,11.97,2.22
5,30회 이상,2761,66.72,12.89,2.14
6,40회 이상,2287,55.27,11.22,1.06
7,50회 이상,1879,45.41,9.96,1.15



===== 초기 지속성 기준 비교 =====


,활동일 기준,사용자,Activated 비중,직전 대비 감소,비중 감소,상대 감소율,W2 격차,W4 격차,W8 격차
0,3일+,3622,87.53,NaN,NaN,NaN,51.12,19.33,2.29
1,4일+,3235,78.18,-387.0,-9.35,-10.68,47.61,19.20,2.42
2,5일+,2740,66.22,-495.0,-11.96,-15.30,45.23,18.97,2.79


In [5]:
# =========================================================
# 최종 Activation 세그먼트 생성
#
# 선정된 기준:
# - 초기 강도 = 첫 Vote 후 24시간 내 Valid Vote 30회 이상
# - 초기 지속성 = R1~R7 중 Valid Vote 활동일 5일 이상
# =========================================================


# ---------------------------------------------------------
# 1. 초기 강도 충족 여부
# ---------------------------------------------------------

may_activation_users['초기강도_활성화'] = (
    may_activation_users['first_vote_24h_votes'] >= 30
)


# ---------------------------------------------------------
# 2. 초기 지속성 충족 여부
# ---------------------------------------------------------

may_activation_users['지속성_활성화'] = (
    may_activation_users['vote_days_R1_7'] >= 5
)


# ---------------------------------------------------------
# 3. 초기 강도 × 초기 지속성으로 4개 세그먼트 생성
# ---------------------------------------------------------

# 기본값: 저강도 + 저지속
may_activation_users['핵심활성화_그룹'] = '저강도저지속'

# 고강도 + 고지속
may_activation_users.loc[
    may_activation_users['초기강도_활성화'] &
    may_activation_users['지속성_활성화'],
    '핵심활성화_그룹'
] = '고강도고지속'

# 고강도 + 저지속
may_activation_users.loc[
    may_activation_users['초기강도_활성화'] &
    ~may_activation_users['지속성_활성화'],
    '핵심활성화_그룹'
] = '고강도저지속'

# 저강도 + 고지속
may_activation_users.loc[
    ~may_activation_users['초기강도_활성화'] &
    may_activation_users['지속성_활성화'],
    '핵심활성화_그룹'
] = '저강도고지속'


# ---------------------------------------------------------
# 4. 결과 확인
# ---------------------------------------------------------

print(
    '1차 Activation 사용자 수:',
    may_activation_users['user_id'].nunique()
)

print('\n세그먼트별 사용자 수')

print(
    may_activation_users[
        '핵심활성화_그룹'
    ].value_counts()
)

1차 Activation 사용자 수: 4138

세그먼트별 사용자 수
핵심활성화_그룹
고강도고지속    2088
저강도저지속     725
고강도저지속     673
저강도고지속     652
Name: count, dtype: int64


In [6]:
# =========================================================
# 세그먼트별 Weekly Retention 분석
#
# W1 = R1~R7
# W2 = R8~R14
# ...
# W8 = R50~R56
# =========================================================


# 세그먼트별 전체 사용자 수
segment_size = (
    may_activation_users
    .groupby('핵심활성화_그룹')['user_id']
    .nunique()
)


# R1~R56 Valid Vote 데이터
weekly_vote = valid_vote_daily[
    (valid_vote_daily['variant'] == 'base') &
    (
        valid_vote_daily['user_id']
        .isin(may_activation_users['user_id'])
    ) &
    (valid_vote_daily['rday'].between(1, 56))
].copy()


# 세그먼트 정보 결합
weekly_vote = weekly_vote.merge(
    may_activation_users[
        ['user_id', '핵심활성화_그룹']
    ],
    on='user_id',
    how='left'
)


# Rday → Week
weekly_vote['week'] = (
    (weekly_vote['rday'] - 1) // 7
) + 1


# 세그먼트 × 주차별 Retained 사용자 수
weekly_retained = (
    weekly_vote
    .groupby(
        ['핵심활성화_그룹', 'week']
    )['user_id']
    .nunique()
    .unstack(fill_value=0)
)

weekly_retained = weekly_retained.reindex(
    columns=range(1, 9),
    fill_value=0
)


# Weekly Retention %
segment_weekly_retention = (
    weekly_retained
    .div(segment_size, axis=0)
    .mul(100)
    .round(2)
)

segment_weekly_retention.columns = [
    f'W{week}'
    for week in range(1, 9)
]


segment_weekly_retention

,W1,W2,W3,W4,W5,W6,W7,W8
핵심활성화_그룹,,,,,,,,
고강도고지속,100.00,83.05,54.02,31.27,17.53,8.52,4.45,3.40
고강도저지속,99.26,38.93,18.42,12.04,6.69,2.38,2.82,1.49
저강도고지속,100.00,84.05,55.67,31.29,17.02,10.28,5.06,4.75
저강도저지속,81.38,37.24,18.62,12.55,6.34,2.76,2.48,0.41


In [7]:
# =========================================================
# 초기 지속성 여부 × 이후 Weekly Retention 카이제곱 검정
#
# W1은 지속성 정의(R1~R7)와 기간이 겹치므로
# 이후 Retention 검정은 W2~W8 사용
# =========================================================

from scipy.stats import chi2_contingency


# ---------------------------------------------------------
# 1. 검증 대상 사용자
# ---------------------------------------------------------

validation_users = may_activation_users[
    [
        'user_id',
        '지속성_활성화',
        '초기강도_활성화'
    ]
].copy()


# ---------------------------------------------------------
# 2. 사용자별 W1~W8 Retention 여부
# ---------------------------------------------------------

weekly_flag = (
    weekly_vote
    .groupby(['user_id', 'week'])
    .size()
    .unstack(fill_value=0)
    .gt(0)
    .astype(int)
)

for week in range(1, 9):

    validation_users[f'W{week}_retained'] = (
        validation_users['user_id']
        .map(
            weekly_flag.get(
                week,
                pd.Series(dtype=int)
            )
        )
        .fillna(0)
        .astype(int)
    )


# ---------------------------------------------------------
# 3. 지속성 충족 여부 × W2~W8 카이제곱 검정
# ---------------------------------------------------------

results = []

for week in range(2, 9):

    col = f'W{week}_retained'

    table = pd.crosstab(
        validation_users['지속성_활성화'],
        validation_users[col]
    )

    chi2, p, dof, expected = chi2_contingency(table)

    rates = (
        validation_users
        .groupby('지속성_활성화')[col]
        .mean()
        .mul(100)
    )

    results.append({
        'Week': f'W{week}',

        '지속성_미충족_Retention(%)':
            rates.get(False, np.nan),

        '지속성_충족_Retention(%)':
            rates.get(True, np.nan),

        '차이(%p)':
            rates.get(True, np.nan)
            - rates.get(False, np.nan),

        'p-value': p
    })


# ---------------------------------------------------------
# 4. 결과표
# ---------------------------------------------------------

weekly_test = (
    pd.DataFrame(results)
    .round(4)
)

weekly_test

,Week,지속성_미충족_Retention(%),지속성_충족_Retention(%),차이(%p),p-value
0,W2,38.0544,83.2847,45.2303,0.000
1,W3,18.5265,54.4161,35.8896,0.000
2,W4,12.3033,31.2774,18.9741,0.000
3,W5,6.5093,17.4088,10.8995,0.000
4,W6,2.5751,8.9416,6.3665,0.000
5,W7,2.6466,4.5985,1.9519,0.003
6,W8,0.9299,3.7226,2.7927,0.000
